In [ ]:
import os
import sys
import numpy as np
from sklearn.metrics import adjusted_mutual_info_score as ami
from sklearn.metrics import adjusted_rand_score as ari
from tqdm import tqdm

HCF_ROOT_FOLDER = "/export/share/peters57dm/Verbund/deepsync/experiments"
os.chdir(HCF_ROOT_FOLDER)
sys.path.append(HCF_ROOT_FOLDER)

from helper.datasets import (
    load_pendigits,
    load_optdigits,
    load_letterrecognition,
    load_gaussian_blobs,
    load_example,
    load_usps,
    load_htru,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    load_mnist,
    load_fmnist,
    load_cifar10,
    load_coil20,
    load_coil100,
    load_cifar100,
    load_weizmann,
)
from helper.deep import (
    Autoencoder,
    detect_device,
    get_train_and_testloader,
    load_pretrained_model,
    encode_batchwise,
)

datasets_loading_methods = [
    load_pendigits,
    load_optdigits,
    load_letterrecognition,
    load_gaussian_blobs,
    load_example,
    load_usps,
    load_htru,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    load_mnist,
    load_fmnist,
    load_cifar10,
    load_coil20,
    load_coil100,
    load_cifar100,
    load_weizmann,
]

In [3]:
from sklearn.neighbors import NearestNeighbors

def find_local_core_points_v1(data, k, percent):
    global indices, refined_medians

    n = data.shape[0]
    subset = int(np.floor(n * percent))

    knn = NearestNeighbors(n_neighbors=subset, metric="euclidean").fit(data)
    distances, indices = knn.kneighbors(data)

    # Compute k-nearest neighbors within each subset
    refined_medians = np.zeros(n)
    for j in range(n):
        j_neighbors = data[indices[j], :]
        knn_j = NearestNeighbors(n_neighbors=k, metric="euclidean").fit(j_neighbors)
        distances_j, _ = knn_j.kneighbors(j_neighbors)
        refined_medians[j] = np.median(distances_j[:, k - 1])

    furthest_neighbor_points_distances = distances[:, k - 1]
    vector_mask = furthest_neighbor_points_distances < refined_medians

    core_points_mask = np.outer(vector_mask, vector_mask).astype(int)
    core_threshold = np.median(refined_medians)

    return core_points_mask, core_threshold

In [4]:
from sklearn.metrics import pairwise_distances

def find_local_core_points_v2(data, k, percent):
    n = data.shape[0]
    subset = int(np.floor(n * percent))

    p_dist = pairwise_distances(data, metric="euclidean")
    core_dists = np.partition(p_dist, k - 1, axis=0)[k - 1]

    nn = np.argpartition(p_dist, subset, axis=1)[:, :subset]
    # nn_mask = np.tile(np.arange(n).reshape(-1, 1), (1, subset))
    # nn = nn[nn != nn_mask].reshape(n, subset - 1)
    refined_medians = np.median(core_dists[nn], axis=1)

    vector_mask = core_dists < refined_medians

    return vector_mask, refined_medians

In [5]:
from sklearn.metrics import pairwise_distances


def find_local_core_points_v3(data, k, percent):
    n = data.shape[0]
    subset = int(np.floor(n * percent))

    p_dist = pairwise_distances(data, metric="euclidean")
    core_dists = np.partition(p_dist, k - 1, axis=0)[k - 1]

    nn = np.argpartition(p_dist, subset, axis=1)[:, :subset]
    # nn_mask = np.tile(np.arange(n).reshape(-1, 1), (1, subset))
    # nn = nn[nn != nn_mask].reshape(n, subset - 1)

    refined_medians = np.empty((n))
    for i in range(n):
        p_dist_neighbors = p_dist[np.ix_(nn[i], nn[i])]
        core_dists_neighbors = np.partition(p_dist_neighbors, k - 1, axis=0)[k - 1]
        refined_medians[i] = np.median(core_dists_neighbors)

    vector_mask = core_dists < refined_medians

    return vector_mask, refined_medians

In [12]:
def load_data_and_embedding(load_fn):
    data, gt_labels, data_name, _ = load_fn()

    PRETRAINED_MODELS_ROOT_PATH = (
        "/export/share/peters57dm/Verbund/deepsync/experiments/comparison102/ae_sync_loss/knn_label_assignment"
    )
    EXP_NO_NAME = "exp_00"
    BATCH_SIZE = 256
    MAX_EMBED_SIZE = 10
    embedded_space_dim = min(data.shape[1], MAX_EMBED_SIZE)

    model = Autoencoder(input_dim=data.shape[1], embedding_size=embedded_space_dim)
    trainloader, testloader = get_train_and_testloader(data, gt_labels, BATCH_SIZE)

    pretrained_model_path = os.path.join(PRETRAINED_MODELS_ROOT_PATH, data_name, EXP_NO_NAME, 'pretrained_autoencoder.pth')
    model = load_pretrained_model(model, pretrained_model_path, device="cpu")
    embedded, gt_labels = encode_batchwise(testloader, model, device="cpu")
    return data, embedded, gt_labels, data_name

In [21]:
from SHiP import SHiP
from SHiP.ultrametric_tree import UltrametricTreeType as UTreeType, AVAILABLE_ULTRAMETRIC_TREE_TYPES
from SHiP.partitioning import PartitioningMethod as PMethod, AVAILABLE_PARTITIONING_METHODS

In [ ]:
excludeTreeTypes = [
    UTreeType.LoadTree,
]
TREE_TYPES = [treeType for treeType in AVAILABLE_ULTRAMETRIC_TREE_TYPES if treeType not in excludeTreeTypes]
HIERACHIES = range(0, 5)
PARTITIONING_METHODS = AVAILABLE_PARTITIONING_METHODS

MIN_POINTS = 5
MIN_CLUSTER_SIZE = 15

for find_core_pts_fn, core_pts_name in [
    (find_local_core_points_v2, "same_core_pts"),
    (find_local_core_points_v3, "adaptive_core_pts"),
]:
    for load_fn in datasets_loading_methods:
        data, embedded_data, gt_labels, data_name = load_data_and_embedding(load_fn)

        core_points_mask, _ = find_core_pts_fn(data, k=50, percent=0.1)

        core_points_original_space = data[core_points_mask]
        core_points_embedding = embedded_data[core_points_mask]
        gt_labels = gt_labels[core_points_mask]

        for core_points, space_name in [
            (core_points_original_space, "original_space"),
            (core_points_embedding, "embedding_space"),
        ]:
            k = len(np.unique(gt_labels))

            print("##########################################")
            print(f"CORE_PTS: {core_pts_name}, SPACE: {space_name}, DATASET: {data_name}, n: {len(core_points)}, dim: {len(core_points[0])}, k: {k}")

            for treeType in TREE_TYPES:
                print(f"Start: {data_name}, {treeType}")
                config = {
                    "k": k,
                    "min_points": MIN_POINTS,
                    "min_cluster_size": MIN_CLUSTER_SIZE,
                    "optimize_tree": True,
                }
                ship = SHiP(data=core_points, treeType=treeType, config=config)

                for power in HIERACHIES:
                    for partitioningMethod in PARTITIONING_METHODS:
                        ship.power = power
                        ship.partitioningMethod = partitioningMethod

                        labels = ship.fit_predict(power, partitioningMethod)

                        savestring = f"./labels/{core_pts_name}##{data_name}##{space_name}##{treeType}##{power}##{partitioningMethod}.npy"
                        os.makedirs(os.path.dirname(savestring), exist_ok=True)
                        np.save(savestring, labels)

In [28]:
import glob

savestring = f"./labels/same_core_pts##weizmann##original_space##UltrametricTreeType.DCTree##"
len(glob.glob(savestring + "*"))

55

In [32]:
excludeTreeTypes = [
    UTreeType.LoadTree,
]
TREE_TYPES = [treeType for treeType in AVAILABLE_ULTRAMETRIC_TREE_TYPES if treeType not in excludeTreeTypes]
HIERACHIES = range(0, 5)
PARTITIONING_METHODS = AVAILABLE_PARTITIONING_METHODS

len(HIERACHIES) * len(PARTITIONING_METHODS)

55